In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd()
if project_root.name == "notebook":
    project_root = project_root.parent

env_path = project_root / ".env"
load_dotenv(env_path, override=True)

print("ENV path:", env_path)
print("ENV exists:", env_path.exists())
print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))

# LangSmith is optional for now because API access is returning 403.
print("LANGSMITH_API_KEY exists:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LANGSMITH_ENDPOINT:", os.getenv("LANGSMITH_ENDPOINT"))
print("LANGSMITH_WORKSPACE_ID:", os.getenv("LANGSMITH_WORKSPACE_ID"))

ENV path: c:\faizul-personal\chatbot-evaluation-langsmith-krishnaik-YT\chatbot-evaluator\.env
ENV exists: True
OPENAI_API_KEY exists: True
LANGSMITH_API_KEY exists: True
LANGSMITH_ENDPOINT: https://aws.api.smith.langchain.com
LANGSMITH_WORKSPACE_ID: 3addb71a-a035-4c04-9335-6aecdeeb74ec


In [7]:
dataset_name = "Chatbots Evaluation"

examples = [
    {
        "inputs": {"question": "What is LangChain?"},
        "outputs": {"answer": "A framework for building LLM applications"},
    },
    {
        "inputs": {"question": "What is LangSmith?"},
        "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
    },
    {
        "inputs": {"question": "What is OpenAI?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    },
    {
        "inputs": {"question": "What is Google?"},
        "outputs": {"answer": "A technology company known for search"},
    },
    {
        "inputs": {"question": "What is Mistral?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    },
    {
        "inputs": {"question": "What is Cefalo?"},
        "outputs": {"answer": "A software company that provides development teams and services"},
    },
]

print(f"Local dataset ready: {dataset_name}")
print(f"Total examples: {len(examples)}")

Local dataset ready: Chatbots Evaluation
Total examples: 6


In [8]:
from openai import OpenAI

openai_client = OpenAI()

In [9]:
default_instructions = "Respond to the user's question in a short, concise manner."

def my_app(
    question: str,
    model: str = "gpt-4o-mini",
    instructions: str = default_instructions,
) -> str:
    response = openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    )

    return response.choices[0].message.content

In [10]:
my_app("What is LangChain?")

'LangChain is a framework designed for building applications that utilize language models. It provides tools and components to facilitate the integration of language models with various data sources, APIs, and workflows, enabling developers to create more complex and interactive applications.'

In [12]:
my_app("What is Cefalo?")

'Cefalo is a technology company that specializes in software development, IT consulting, and digital transformation services. They focus on delivering innovative solutions to enhance business processes and improve efficiency.'

In [13]:
my_app("What is cricket?")

'Cricket is a bat-and-ball sport played between two teams of eleven players each. It involves batting, bowling, and fielding, with the objective of scoring runs and dismissing the opposing team. The game is played on a circular or oval field, with a 22-yard long pitch at the center.'

In [15]:
###define evaluator 

eval_instructions = "You are an expert evaluator. Grade whether the predicted answer is semantically correct compared to the reference answer."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    predicted_answer = outputs.get("response") or outputs.get("answer")

    user_content = f"""
Question:
{inputs['question']}

Reference answer:
{reference_outputs['answer']}

Predicted answer:
{predicted_answer}

Is the predicted answer semantically correct compared to the reference answer?

Respond with only one word:
CORRECT or INCORRECT
"""

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content},
        ],
    )

    verdict = response.choices[0].message.content.strip().upper()
    return verdict == "CORRECT"


def concision(inputs: dict, outputs: dict, reference_outputs: dict = None) -> bool:
    predicted_answer = outputs.get("response") or outputs.get("answer")
    return len(predicted_answer.split()) <= 30

In [18]:
### Run local evaluation
import pandas as pd

local_results = []

for example in examples:
    inputs = example["inputs"]
    reference_outputs = example["outputs"]

    model_response = my_app(inputs["question"])

    outputs = {
        "response": model_response
    }

    result = {
        "question": inputs["question"],
        "reference_answer": reference_outputs["answer"],
        "model_response": model_response,
        "correctness": correctness(inputs, outputs, reference_outputs),
        "concision": concision(inputs, outputs, reference_outputs),
    }

    local_results.append(result)

results_df = pd.DataFrame(local_results)
results_df

,question,reference_answer,model_response,correctness,concision
0,What is LangChain?,A framework for building LLM applications,LangChain is a framework designed for building...,True,False
1,What is LangSmith?,A platform for observing and evaluating LLM ap...,LangSmith is a platform designed for developer...,True,False
2,What is OpenAI?,A company that creates Large Language Models,OpenAI is an artificial intelligence research ...,True,False
3,What is Google?,A technology company known for search,Google is a multinational technology company s...,True,False
4,What is Mistral?,A company that creates Large Language Models,Mistral is a company that develops advanced AI...,True,False
5,What is Cefalo?,A software company that provides development t...,Cefalo is a technology company that specialize...,True,True
